In [ ]:
#Import neccessary libraries
import os
os.environ["GIT_PYTHON_REFRESH"] = "quiet"
import git

from myosuite.utils import gym
import skvideo
skvideo.setFFmpegPath(r"C:\Users\rohan\Downloads\ffmpeg-2025-09-04-git-2611874a50-essentials_build\ffmpeg-2025-09-04-git-2611874a50-essentials_build\bin")
import skvideo.io

import numpy as np
import os

In [ ]:
#5.0 Combined ExoOnlyWrapper
import myosuite
from myosuite.utils import gym
import numpy as np
from stable_baselines3 import PPO


class CombinedExoOnlyWrapper(gym.Env):
    def __init__(self, base_env, frozen_policy_path, 
                 
                 smart_reset=False, use_avg_mf=False, 
                 
                 force_scale_range=(0.8, 0.9), activation_slowdown_range=(2.0, 3.0),
                 
                 hide_pose_err=False):
        super(CombinedExoOnlyWrapper, self).__init__()
        
        self.base_env = base_env
        self.frozen_policy = PPO.load(frozen_policy_path)
        
        
        self.action_space = gym.spaces.Box(low=np.array([0.0]), high=np.array([1.0]), dtype=np.float32)
        
        
        self.smart_reset = smart_reset
        self.use_avg_mf = use_avg_mf
        self.n_muscles = len(self.base_env.unwrapped.muscle_fatigue.MA)
        
        
        self.force_scale_range = force_scale_range
        self.activation_slowdown_range = activation_slowdown_range
        self.force_scale = 1.0
        self.activation_slowdown = 1.0
        self.original_gear = None
        self.original_dynprm = None
        
        
        self.hide_pose_err = hide_pose_err
        
        
        self.exo_obs_space = self.base_env.observation_space
        self.base_obs_dim = self.frozen_policy.observation_space.shape[0]
        self.exo_obs_dim = self.exo_obs_space.shape[0]
        


        # Detect pose_err dimension
        sample_obs_dict = self.base_env.unwrapped.get_obs_dict(self.base_env.unwrapped.sim)
        pose_err_sample = sample_obs_dict.get("pose_err", np.array([]))
        if hasattr(pose_err_sample, '__len__') and not isinstance(pose_err_sample, str):
            self.pose_err_dim = len(pose_err_sample)
        else:
            self.pose_err_dim = 1
        
        
        
        if self.hide_pose_err:
            base_obs_dim_adjusted = self.base_obs_dim - self.pose_err_dim
            base_obs_low = self.exo_obs_space.low[:base_obs_dim_adjusted]
            base_obs_high = self.exo_obs_space.high[:base_obs_dim_adjusted]
        else:
            base_obs_low = self.exo_obs_space.low[:self.base_obs_dim]
            base_obs_high = self.exo_obs_space.high[:base_obs_dim]
            base_obs_dim_adjusted = self.base_obs_dim
        
        
        if self.use_avg_mf:
            
            mf_low = np.array([0.0], dtype=np.float32)
            mf_high = np.array([1.0], dtype=np.float32)
            mf_desc = "avg MF (1D)"
            mf_dim = 1
        else:
            
            mf_low = np.zeros(self.n_muscles, dtype=np.float32)
            mf_high = np.ones(self.n_muscles, dtype=np.float32)
            mf_desc = f"MF values ({self.n_muscles}D)"
            mf_dim = self.n_muscles
        
        
        brady_low = np.array([self.force_scale_range[0], self.activation_slowdown_range[0]], dtype=np.float32)
        brady_high = np.array([self.force_scale_range[1], self.activation_slowdown_range[1]], dtype=np.float32)
        brady_desc = "bradykinesia (2D)"
        
        
        combined_low = np.concatenate([base_obs_low, mf_low, brady_low])
        combined_high = np.concatenate([base_obs_high, mf_high, brady_high])
        
        self.observation_space = gym.spaces.Box(
            low=combined_low,
            high=combined_high,
            dtype=np.float32
        )
        
        total_dim = base_obs_dim_adjusted + mf_dim + 2
        obs_desc = f"base obs ({base_obs_dim_adjusted}D) + {mf_desc} + {brady_desc} = {total_dim}D"
        
        
        if not smart_reset:
            self.base_env.unwrapped.set_fatigue_reset_random(True)

    def _sample_bradykinesia_parameters(self):
        
        self.force_scale = np.random.uniform(*self.force_scale_range)
        self.activation_slowdown = np.random.uniform(*self.activation_slowdown_range)
    
    def _apply_bradykinesia_effects(self):
        
        model = self.base_env.unwrapped.sim.model
        
       
        if self.original_gear is None:
            self.original_gear = model.actuator_gear.copy()

        
        model.actuator_gear[:] = self.original_gear * self.force_scale

        
        try:
            if hasattr(model, 'actuator_dynprm') and model.actuator_dynprm.size > 0:
                if self.original_dynprm is None:
                    self.original_dynprm = model.actuator_dynprm.copy()
                model.actuator_dynprm[:] = self.original_dynprm * self.activation_slowdown
        except Exception as e:
            print(f" fix")
    
    def _restore_original_parameters(self):
        
        if self.original_gear is not None:
            model = self.base_env.unwrapped.sim.model
            model.actuator_gear[:] = self.original_gear
            
            if self.original_dynprm is not None:
                try:
                    model.actuator_dynprm[:] = self.original_dynprm
                except:
                    pass

    def _get_extended_obs(self, exo_obs):

        if self.hide_pose_err:
            base_obs_dim_adjusted = self.base_obs_dim - self.pose_err_dim
            base_obs = exo_obs[:base_obs_dim_adjusted].astype(np.float32)
        else:
            base_obs = exo_obs[:self.base_obs_dim].astype(np.float32)
        
    
        mf_values = self.base_env.unwrapped.muscle_fatigue.MF.copy().astype(np.float32)
        
        if self.use_avg_mf:
            
            mf_obs = np.array([np.mean(mf_values)], dtype=np.float32)
        else:
            
            mf_obs = mf_values
        
       
        brady_obs = np.array([self.force_scale, self.activation_slowdown], dtype=np.float32)
        
       
        extended_obs = np.concatenate([base_obs, mf_obs, brady_obs])
        
        return extended_obs

    def _get_base_obs_for_policy(self, exo_obs):
        
        return exo_obs[:self.base_obs_dim].astype(np.float32)

    def reset(self, **kwargs):
        
        self._sample_bradykinesia_parameters()
        
        
        exo_obs, info = self.base_env.reset(fatigue_reset=True, **kwargs)
        
       
        try:
            low = self.base_env.unwrapped.target_jnt_range[:, 0]
            high = self.base_env.unwrapped.target_jnt_range[:, 1]
            new_target = np.random.uniform(low, high)
            self.base_env.unwrapped.target_jnt_value = new_target
            self.base_env.unwrapped.target_type = 'fixed'
        except Exception as e:
            print(f"  fix {e}")

        
        if self.smart_reset:
            
            MF = np.random.uniform(0.7, 1.0, size=self.n_muscles)
            remaining = 1.0 - MF
            split = np.random.uniform(0.0, 1.0, size=self.n_muscles)
            MA = remaining * split
            MR = remaining * (1.0 - split)

            
            self.base_env.unwrapped.muscle_fatigue.MA[:] = MA
            self.base_env.unwrapped.muscle_fatigue.MR[:] = MR
            self.base_env.unwrapped.muscle_fatigue.MF[:] = MF

        
        self._apply_bradykinesia_effects()

        
        extended_obs = self._get_extended_obs(exo_obs)
        return extended_obs, info

    def step(self, exo_action):
        """Execute one step with exoskeleton action"""
        exo_action = np.array([exo_action], dtype=np.float32).reshape(-1)

        
        try:
            current_exo_obs = self.base_env._get_obs()
        except AttributeError:
            current_exo_obs = self.base_env.unwrapped.get_obs()

        
        base_obs_for_policy = self._get_base_obs_for_policy(current_exo_obs)

       
        muscle_actions, _ = self.frozen_policy.predict(base_obs_for_policy, deterministic=True)

        full_action = np.concatenate([exo_action, muscle_actions])

        next_exo_obs, reward, done, truncated, info = self.base_env.step(full_action)

        extended_obs = self._get_extended_obs(next_exo_obs)
        return extended_obs, reward, done, truncated, info

    def render(self, *args, **kwargs):
        return self.base_env.render(*args, **kwargs)

    def close(self):
        self._restore_original_parameters()
        self.base_env.close()
    
    def get_fatigue_info(self):
        mf_values = self.base_env.unwrapped.muscle_fatigue.MF.copy()
        ma_values = self.base_env.unwrapped.muscle_fatigue.MA.copy()
        mr_values = self.base_env.unwrapped.muscle_fatigue.MR.copy()
        
        return {
            'muscle_fatigue': mf_values,
            'muscle_activation': ma_values,
            'muscle_recovery': mr_values,
            'avg_fatigue': np.mean(mf_values)
        }
    
    def get_bradykinesia_info(self):
        return {
            'force_scale': self.force_scale,
            'activation_slowdown': self.activation_slowdown
        }
    
    def get_combined_info(self):
        fatigue_info = self.get_fatigue_info()
        brady_info = self.get_bradykinesia_info()
        
        return {
            'fatigue': fatigue_info,
            'bradykinesia': brady_info,
            'observation_breakdown': {
                'base_obs_dim': self.base_obs_dim - (self.pose_err_dim if self.hide_pose_err else 0),
                'fatigue_obs_dim': 1 if self.use_avg_mf else self.n_muscles,
                'bradykinesia_obs_dim': 2,
                'total_obs_dim': self.observation_space.shape[0]
            }
        }

In [ ]:
#FINAL EVAL

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import time
import myosuite
from myosuite.utils import gym
import skvideo.io
from stable_baselines3 import PPO
from IPython.display import HTML, display
from base64 import b64encode
import PIL.Image, PIL.ImageDraw
from scipy import signal


EPI_NUM = 3
env_name_exo = "myoFatiElbowPose1D6MExoRandom-v0"
GENERATE_VIDEO = True
max_steps = 500

COLOR_PD_EXO = '#E74C3C'
COLOR_HEALTHY = '#3498DB'
COLOR_PD_NOEXO = '#95A5A6'


print(" PARKINSON'S IMPAIRMENT EXOSKELETON EVALUATION")


def show_video(video_path, video_width=800):
    video_file = open(video_path, "r+b").read()
    video_url = f'data:video/mp4;base64,{b64encode(video_file).decode()}'
    return HTML(f"<video autoplay width={video_width} controls><source src=\"{video_url}\"></video>")

def add_text_to_frame(frame, text, pos=(40, 40), color=(255, 0, 0), fontsize=12):
    if isinstance(frame, np.ndarray):
        frame = PIL.Image.fromarray(frame)
    draw = PIL.ImageDraw.Draw(frame)
    draw.text(pos, text, fill=color)
    return np.asarray(frame)

def compute_jerk(positions, dt):
    velocity = np.gradient(positions, dt)
    acceleration = np.gradient(velocity, dt)
    jerk = np.gradient(acceleration, dt)
    return jerk

def compute_power(torque, velocity):
    return torque * velocity

def detect_movement_initiation(velocity, threshold=0.15):
    for i, vel in enumerate(velocity):
        if abs(vel) > threshold:
            return i
    return len(velocity)

def safe_pose_error(x):
    err_val = x['pose_err'].iloc[-1]
    if hasattr(err_val, '__len__'):
        return np.mean(np.abs(err_val))
    else:
        return np.abs(err_val)


exo_policy = PPO.load("Full-ElbowExo")
human_policy = PPO.load("ElbowPose_policy")
print("✓ Models loaded successfully")


all_data = {'PD+Exo': [], 'PD-NoExo': [], 'Healthy': []}
latency_data = []
all_videos = {'PD+Exo': [], 'PD-NoExo': [], 'Healthy': []}


for epi_set in range(EPI_NUM):
    print(f"\n{'='*80}")
    print(f"EVAL SET {epi_set + 1}/{EPI_NUM}")
    print(f"{'='*80}")
    
    initial_qpos_val = 0.0
    initial_qvel_val = 0.0
    target_angle = np.random.uniform(0.5, 1.5)
    
    print(f"Set {epi_set + 1} Configuration:")
    print(f"  Initial Position: {initial_qpos_val:.3f} rad")
    print(f"  Target Position: {target_angle:.3f} rad")
    
    # PD + EXOSKELETON
    print(f"\n--- Condition 1/3: PD + Exoskeleton Assistance ---")
    base_env_pd_exo = gym.make(env_name_exo)
    pd_exo_env = CombinedExoOnlyWrapper(base_env_pd_exo, frozen_policy_path="ElbowPose_policy",
                                        smart_reset=True, use_avg_mf=True, hide_pose_err=True)
    
    obs, _ = pd_exo_env.reset()
    pd_exo_env.base_env.unwrapped.target_jnt_value = [target_angle]
    pd_exo_env.base_env.unwrapped.target_type = 'fixed'
    pd_exo_env.base_env.unwrapped.update_target(restore_sim=True)
    pd_exo_env.base_env.unwrapped.sim.data.qpos[:] = [initial_qpos_val]
    pd_exo_env.base_env.unwrapped.sim.data.qvel[:] = [initial_qvel_val]
    pd_exo_env.base_env.unwrapped.sim.forward()
    
    n_muscles = len(pd_exo_env.base_env.unwrapped.muscle_fatigue.MA)
    MF_fixed = np.random.uniform(0.5, 0.9, size=n_muscles)
    remaining = 1.0 - MF_fixed
    split = np.random.uniform(0.0, 1.0, size=n_muscles)
    MA_fixed = remaining * split
    MR_fixed = remaining * (1.0 - split)
    
    pd_exo_env.base_env.unwrapped.muscle_fatigue.MA[:] = MA_fixed
    pd_exo_env.base_env.unwrapped.muscle_fatigue.MR[:] = MR_fixed
    pd_exo_env.base_env.unwrapped.muscle_fatigue.MF[:] = MF_fixed
    
    brady_info = pd_exo_env.get_bradykinesia_info()
    print(f"  Avg Fatigue: {np.mean(MF_fixed):.3f}")
    
    episode_data = []
    episode_frames = []
    tr = 0.0
    
    for step in range(max_steps):
        if GENERATE_VIDEO:
            frame = pd_exo_env.base_env.unwrapped.sim.renderer.render_offscreen(width=400, height=400, camera_id=0)
            current_time = step * pd_exo_env.base_env.unwrapped.dt
            frame = add_text_to_frame(frame, f"PD+Exo Set{epi_set+1} t={current_time:.1f}s", 
                                     pos=(50, 3), color=(255, 0, 0))
            episode_frames.append(frame.astype(np.uint8))
        
        start_time = time.perf_counter()
        exo_action, _ = exo_policy.predict(obs, deterministic=True)
        end_time = time.perf_counter()
        latency_ms = (end_time - start_time) * 1000
        latency_data.append({'set': epi_set, 'step': step, 'latency_ms': latency_ms})
        
        next_obs, reward, done, truncated, info = pd_exo_env.step(exo_action)
        obs = next_obs
        tr += float(reward)
        
        fatigue_info = pd_exo_env.get_fatigue_info()
        brady_info_step = pd_exo_env.get_bradykinesia_info()
        
        episode_data.append({
            'set': epi_set, 'condition': 'PD+Exo', 'step': step,
            'time': step * pd_exo_env.base_env.unwrapped.dt,
            'jpos': pd_exo_env.base_env.unwrapped.sim.data.qpos[0],
            'jvel': pd_exo_env.base_env.unwrapped.sim.data.qvel[0],
            'exo_torque': float(exo_action[0]) if len(exo_action) > 0 else 0.0,
            'exo_action_mag': np.linalg.norm(exo_action),
            'mlen': pd_exo_env.base_env.unwrapped.sim.data.actuator_length.copy(),
            'act': pd_exo_env.base_env.unwrapped.sim.data.act.copy(),
            'ctrl': pd_exo_env.base_env.unwrapped.last_ctrl.copy(),
            'reward': float(reward), 'cumulative_reward': tr,
            'solved': pd_exo_env.base_env.unwrapped.rwd_dict['solved'].item(),
            'pose_err': pd_exo_env.base_env.unwrapped.get_obs_dict(pd_exo_env.base_env.unwrapped.sim)["pose_err"],
            'MA': fatigue_info['muscle_activation'].copy(),
            'MR': fatigue_info['muscle_recovery'].copy(),
            'MF': fatigue_info['muscle_fatigue'].copy(),
            'avg_fatigue': fatigue_info['avg_fatigue'],
            'force_scale': brady_info_step['force_scale'],
            'activation_slowdown': brady_info_step['activation_slowdown']
        })
        
        if done or truncated:
            break
    
    all_data['PD+Exo'].extend(episode_data)
    if episode_frames:
        all_videos['PD+Exo'].append(episode_frames)
    print(f"  ✓ Completed: {len(episode_data)} steps, Reward: {tr:.3f}")
    pd_exo_env.close()
    
    # PD WITHOUT EXOSKELETON
    print(f"\n--- Condition 2/3: PD Without Exoskeleton ---")
    base_env_pd_noexo = gym.make(env_name_exo)
    pd_noexo_env = CombinedExoOnlyWrapper(base_env_pd_noexo, frozen_policy_path="ElbowPose_policy",
                                          smart_reset=True, use_avg_mf=True, hide_pose_err=True)
    
    obs, _ = pd_noexo_env.reset()
    pd_noexo_env.base_env.unwrapped.target_jnt_value = [target_angle]
    pd_noexo_env.base_env.unwrapped.target_type = 'fixed'
    pd_noexo_env.base_env.unwrapped.update_target(restore_sim=True)
    pd_noexo_env.base_env.unwrapped.sim.data.qpos[:] = [initial_qpos_val]
    pd_noexo_env.base_env.unwrapped.sim.data.qvel[:] = [initial_qvel_val]
    pd_noexo_env.base_env.unwrapped.sim.forward()
    
    pd_noexo_env.base_env.unwrapped.muscle_fatigue.MA[:] = MA_fixed
    pd_noexo_env.base_env.unwrapped.muscle_fatigue.MR[:] = MR_fixed
    pd_noexo_env.base_env.unwrapped.muscle_fatigue.MF[:] = MF_fixed
    
    episode_data = []
    episode_frames = []
    tr = 0.0
    
    for step in range(max_steps):
        if GENERATE_VIDEO:
            frame = pd_noexo_env.base_env.unwrapped.sim.renderer.render_offscreen(width=400, height=400, camera_id=0)
            current_time = step * pd_noexo_env.base_env.unwrapped.dt
            frame = add_text_to_frame(frame, f"PD-NoExo Set{epi_set+1} t={current_time:.1f}s", 
                                     pos=(50, 3), color=(150, 150, 150))
            episode_frames.append(frame.astype(np.uint8))
        
        exo_action = np.array([0.0], dtype=np.float32)
        next_obs, reward, done, truncated, info = pd_noexo_env.step(exo_action)
        obs = next_obs
        tr += float(reward)
        
        fatigue_info = pd_noexo_env.get_fatigue_info()
        brady_info_step = pd_noexo_env.get_bradykinesia_info()
        
        episode_data.append({
            'set': epi_set, 'condition': 'PD-NoExo', 'step': step,
            'time': step * pd_noexo_env.base_env.unwrapped.dt,
            'jpos': pd_noexo_env.base_env.unwrapped.sim.data.qpos[0],
            'jvel': pd_noexo_env.base_env.unwrapped.sim.data.qvel[0],
            'exo_torque': 0.0, 'exo_action_mag': 0.0,
            'mlen': pd_noexo_env.base_env.unwrapped.sim.data.actuator_length.copy(),
            'act': pd_noexo_env.base_env.unwrapped.sim.data.act.copy(),
            'ctrl': pd_noexo_env.base_env.unwrapped.last_ctrl.copy(),
            'reward': float(reward), 'cumulative_reward': tr,
            'solved': pd_noexo_env.base_env.unwrapped.rwd_dict['solved'].item(),
            'pose_err': pd_noexo_env.base_env.unwrapped.get_obs_dict(pd_noexo_env.base_env.unwrapped.sim)["pose_err"],
            'MA': fatigue_info['muscle_activation'].copy(),
            'MR': fatigue_info['muscle_recovery'].copy(),
            'MF': fatigue_info['muscle_fatigue'].copy(),
            'avg_fatigue': fatigue_info['avg_fatigue'],
            'force_scale': brady_info_step['force_scale'],
            'activation_slowdown': brady_info_step['activation_slowdown']
        })
        
        if done or truncated:
            break
    
    all_data['PD-NoExo'].extend(episode_data)
    if episode_frames:
        all_videos['PD-NoExo'].append(episode_frames)
    print(f"  ✓ Completed: {len(episode_data)} steps, Reward: {tr:.3f}")
    pd_noexo_env.close()
    
    # HEALTHY PATIENT
    print(f"\n--- Condition 3/3: Healthy Subject ---")
    base_env_healthy = gym.make(env_name_exo)
    
    obs, _ = base_env_healthy.reset()
    base_env_healthy.unwrapped.target_jnt_value = [target_angle]
    base_env_healthy.unwrapped.target_type = 'fixed'
    base_env_healthy.unwrapped.update_target(restore_sim=True)
    base_env_healthy.unwrapped.sim.data.qpos[:] = [initial_qpos_val]
    base_env_healthy.unwrapped.sim.data.qvel[:] = [initial_qvel_val]
    base_env_healthy.unwrapped.sim.forward()
    
    episode_data = []
    episode_frames = []
    tr = 0.0
    
    for step in range(max_steps):
        if GENERATE_VIDEO:
            frame = base_env_healthy.unwrapped.sim.renderer.render_offscreen(width=400, height=400, camera_id=0)
            current_time = step * base_env_healthy.unwrapped.dt
            frame = add_text_to_frame(frame, f"Healthy Set{epi_set+1} t={current_time:.1f}s", 
                                     pos=(50, 3), color=(0, 0, 255))
            episode_frames.append(frame.astype(np.uint8))
        
        current_obs = base_env_healthy.unwrapped.get_obs()
        human_action, _ = human_policy.predict(current_obs, deterministic=True)
        full_action = np.concatenate([np.array([0.0]), human_action], dtype=np.float32)
        next_obs, reward, done, truncated, info = base_env_healthy.step(full_action)
        tr += float(reward)
        
        episode_data.append({
            'set': epi_set, 'condition': 'Healthy', 'step': step,
            'time': step * base_env_healthy.unwrapped.dt,
            'jpos': base_env_healthy.unwrapped.sim.data.qpos[0],
            'jvel': base_env_healthy.unwrapped.sim.data.qvel[0],
            'exo_torque': 0.0, 'exo_action_mag': 0.0,
            'mlen': base_env_healthy.unwrapped.sim.data.actuator_length.copy(),
            'act': base_env_healthy.unwrapped.sim.data.act.copy(),
            'ctrl': base_env_healthy.unwrapped.last_ctrl.copy(),
            'reward': float(reward), 'cumulative_reward': tr,
            'solved': base_env_healthy.unwrapped.rwd_dict['solved'].item(),
            'pose_err': base_env_healthy.unwrapped.get_obs_dict(base_env_healthy.unwrapped.sim)["pose_err"],
            'MA': base_env_healthy.unwrapped.muscle_fatigue.MA.copy() if hasattr(base_env_healthy.unwrapped, 'muscle_fatigue') else np.zeros(6),
            'MR': base_env_healthy.unwrapped.muscle_fatigue.MR.copy() if hasattr(base_env_healthy.unwrapped, 'muscle_fatigue') else np.zeros(6),
            'MF': base_env_healthy.unwrapped.muscle_fatigue.MF.copy() if hasattr(base_env_healthy.unwrapped, 'muscle_fatigue') else np.ones(6),
            'avg_fatigue': np.mean(base_env_healthy.unwrapped.muscle_fatigue.MF) if hasattr(base_env_healthy.unwrapped, 'muscle_fatigue') else 1.0,
            'force_scale': 1.0,
            'activation_slowdown': 1.0
        })
        
        if done or truncated:
            break
    
    all_data['Healthy'].extend(episode_data)
    if episode_frames:
        all_videos['Healthy'].append(episode_frames)
    print(f"  ✓ Completed: {len(episode_data)} steps, Reward: {tr:.3f}")
    base_env_healthy.close()


if GENERATE_VIDEO:
    print(f"\n{'='*80}")
    print("SAVING VIDEOS")
    print(f"{'='*80}")
    os.makedirs('videos', exist_ok=True)
    for condition, video_list in all_videos.items():
        for idx, frames in enumerate(video_list):
            if frames:
                video_path = f'videos/{condition}_set{idx+1}.mp4'
                skvideo.io.vwrite(video_path, np.asarray(frames).astype(np.uint8), 
                                outputdict={'-pix_fmt': 'yuv420p'})
                print(f"  ✓ {video_path}")



df_pd_exo = pd.DataFrame(all_data['PD+Exo'])
df_pd_noexo = pd.DataFrame(all_data['PD-NoExo'])
df_healthy = pd.DataFrame(all_data['Healthy'])
df_latency = pd.DataFrame(latency_data)

print(f"  PD+Exo episodes: {len(df_pd_exo)}")
print(f"  PD-NoExo episodes: {len(df_pd_noexo)}")
print(f"  Healthy episodes: {len(df_healthy)}")


avg_latency = df_latency['latency_ms'].mean()
print(f"\n  Average Model Latency: {avg_latency:.4f} ms")


os.makedirs('data', exist_ok=True)
df_pd_exo.to_csv('data/pd_exo_data.csv', index=False)
df_pd_noexo.to_csv('data/pd_noexo_data.csv', index=False)
df_healthy.to_csv('data/healthy_data.csv', index=False)
df_latency.to_csv('data/latency_data.csv', index=False)

# Create summary metrics
summary_metrics = {
    'Condition': ['PD+Exo', 'PD-NoExo', 'Healthy'],
    'Total_Episodes': [len(df_pd_exo), len(df_pd_noexo), len(df_healthy)],
    'Avg_Cumulative_Reward': [
        df_pd_exo.groupby('set')['cumulative_reward'].last().mean(),
        df_pd_noexo.groupby('set')['cumulative_reward'].last().mean(),
        df_healthy.groupby('set')['cumulative_reward'].last().mean()
    ],
    'Avg_Final_Pose_Error': [
        df_pd_exo.groupby('set')['pose_err'].last().mean(),
        df_pd_noexo.groupby('set')['pose_err'].last().mean(),
        df_healthy.groupby('set')['pose_err'].last().mean()
    ],
    'Task_Success_Rate': [
        df_pd_exo.groupby('set')['solved'].last().mean() * 100,
        df_pd_noexo.groupby('set')['solved'].last().mean() * 100,
        df_healthy.groupby('set')['solved'].last().mean() * 100
    ]
}
df_summary = pd.DataFrame(summary_metrics)
df_summary.to_csv('data/summary_metrics.csv', index=False)


os.makedirs('plots', exist_ok=True)

# Set plot style
plt.style.use('default')
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 9

# ============================================================================
# GRAPH 1: MODEL LATENCY ANALYSIS
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df_latency['step'], df_latency['latency_ms'], 
        color='#3498DB', alpha=0.6, linewidth=0.8, label='Per-Step Latency')
ax.axhline(y=avg_latency, color='#E74C3C', linestyle='--', linewidth=2, 
          label=f'Average: {avg_latency:.4f} ms')

# Add text annotation for average
ax.text(len(df_latency) * 0.7, avg_latency * 1.1, 
       f'Avg = {avg_latency:.4f} ms', 
       fontsize=11, color='#E74C3C', fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('Timestep', fontweight='bold')
ax.set_ylabel('Latency (ms)', fontweight='bold')
ax.set_title('PPO Model Inference Latency (PD+Exo Condition)', fontweight='bold', pad=15)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/01_model_latency.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

# Plot each condition
for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    alpha_val = 0.7 if EPI_NUM == 1 else 0.4
    
    ax.plot(df_pd_exo_set['time'], df_pd_exo_set['jpos'], 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
           label='PD+Exo' if set_num == 0 else '')
    ax.plot(df_pd_noexo_set['time'], df_pd_noexo_set['jpos'], 
           color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
           label='PD-NoExo' if set_num == 0 else '')
    ax.plot(df_healthy_set['time'], df_healthy_set['jpos'], 
           color=COLOR_HEALTHY, alpha=alpha_val, linewidth=2, 
           label='Healthy' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Elbow Joint Position (rad)', fontweight='bold')
ax.set_title('Elbow Joint Position Trajectory Comparison\nAcross Three Conditions', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/02_joint_position_comparison.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    alpha_val = 0.7 if EPI_NUM == 1 else 0.4
    
    ax.plot(df_pd_exo_set['time'], df_pd_exo_set['jvel'], 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
           label='PD+Exo' if set_num == 0 else '')
    ax.plot(df_pd_noexo_set['time'], df_pd_noexo_set['jvel'], 
           color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
           label='PD-NoExo' if set_num == 0 else '')
    ax.plot(df_healthy_set['time'], df_healthy_set['jvel'], 
           color=COLOR_HEALTHY, alpha=alpha_val, linewidth=2, 
           label='Healthy' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Angular Velocity (rad/s)', fontweight='bold')
ax.set_title('Joint Velocity Profile Comparison\nDemonstrating Movement Smoothness', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/03_joint_velocity_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    alpha_val = 0.7 if EPI_NUM == 1 else 0.4
    
    ax.plot(df_pd_exo_set['time'], np.abs(df_pd_exo_set['pose_err']), 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
           label='PD+Exo' if set_num == 0 else '')
    ax.plot(df_pd_noexo_set['time'], np.abs(df_pd_noexo_set['pose_err']), 
           color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
           label='PD-NoExo' if set_num == 0 else '')
    ax.plot(df_healthy_set['time'], np.abs(df_healthy_set['pose_err']), 
           color=COLOR_HEALTHY, alpha=alpha_val, linewidth=2, 
           label='Healthy' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Position Error (rad)', fontweight='bold')
ax.set_title('Position Error Convergence Over Time\nLower is Better', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/04_position_error_convergence.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    alpha_val = 0.7 if EPI_NUM == 1 else 0.4
    
    ax.plot(df_pd_exo_set['time'], df_pd_exo_set['cumulative_reward'], 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
           label='PD+Exo' if set_num == 0 else '')
    ax.plot(df_pd_noexo_set['time'], df_pd_noexo_set['cumulative_reward'], 
           color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
           label='PD-NoExo' if set_num == 0 else '')
    ax.plot(df_healthy_set['time'], df_healthy_set['cumulative_reward'], 
           color=COLOR_HEALTHY, alpha=alpha_val, linewidth=2, 
           label='Healthy' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Cumulative Reward', fontweight='bold')
ax.set_title('Cumulative Reward Progression\nDemonstrating Task Performance Quality', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/05_cumulative_reward.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    alpha_val = 0.7 if EPI_NUM == 1 else 0.5
    
    ax.plot(df_pd_exo_set['time'], df_pd_exo_set['exo_torque'], 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
           label=f'Set {set_num+1}' if EPI_NUM > 1 else 'Exo Torque')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Exoskeleton Torque (Nm)', fontweight='bold')
ax.set_title('Exoskeleton Torque Output Over Time\nPPO Model Control Signal', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.savefig('plots/06_exoskeleton_torque.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    
    alpha_val = 0.7 if EPI_NUM == 1 else 0.4
    
    ax.plot(df_pd_exo_set['time'], df_pd_exo_set['avg_fatigue'], 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
           label='PD+Exo' if set_num == 0 else '')
    ax.plot(df_pd_noexo_set['time'], df_pd_noexo_set['avg_fatigue'], 
           color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
           label='PD-NoExo' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Average Muscle Fatigue', fontweight='bold')
ax.set_title('Muscle Fatigue Progression Comparison\nExoskeleton Reduces Fatigue Accumulation', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/07_muscle_fatigue_comparison.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    alpha_val = 0.7 if EPI_NUM == 1 else 0.4
    
    ax.plot(df_pd_exo_set['time'], df_pd_exo_set['force_scale'], 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
           label='PD+Exo' if set_num == 0 else '')
    ax.plot(df_pd_noexo_set['time'], df_pd_noexo_set['force_scale'], 
           color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
           label='PD-NoExo' if set_num == 0 else '')
    ax.plot(df_healthy_set['time'], df_healthy_set['force_scale'], 
           color=COLOR_HEALTHY, alpha=alpha_val, linewidth=2, linestyle='--',
           label='Healthy' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Force Scale Factor', fontweight='bold')
ax.set_title('Bradykinesia-Induced Force Reduction\n(Lower = More Severe Impairment)', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/08_bradykinesia_force_scale.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    
    alpha_val = 0.7 if EPI_NUM == 1 else 0.4
    
    ax.plot(df_pd_exo_set['time'], df_pd_exo_set['activation_slowdown'], 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
           label='PD+Exo' if set_num == 0 else '')
    ax.plot(df_pd_noexo_set['time'], df_pd_noexo_set['activation_slowdown'], 
           color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
           label='PD-NoExo' if set_num == 0 else '')

ax.axhline(y=1.0, color=COLOR_HEALTHY, linestyle='--', linewidth=2, 
          alpha=0.7, label='Healthy Baseline')
ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Activation Slowdown Factor (×)', fontweight='bold')
ax.set_title('Muscle Activation Speed Impairment\n(Higher = Slower Muscle Response)', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/09_activation_slowdown.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    alpha_val = 0.7 if EPI_NUM == 1 else 0.4
    
    ax.plot(df_pd_exo_set['time'], df_pd_exo_set['reward'], 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=1.5, 
           label='PD+Exo' if set_num == 0 else '')
    ax.plot(df_pd_noexo_set['time'], df_pd_noexo_set['reward'], 
           color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=1.5, 
           label='PD-NoExo' if set_num == 0 else '')
    ax.plot(df_healthy_set['time'], df_healthy_set['reward'], 
           color=COLOR_HEALTHY, alpha=alpha_val, linewidth=1.5, 
           label='Healthy' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Instantaneous Reward', fontweight='bold')
ax.set_title('Instantaneous Reward Signal Over Time\nReflects Moment-to-Moment Performance', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/10_instantaneous_reward.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    alpha_val = 0.7 if EPI_NUM == 1 else 0.4
    
    ax.plot(df_pd_exo_set['time'], df_pd_exo_set['solved'], 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
           label='PD+Exo' if set_num == 0 else '', marker='o', markersize=2)
    ax.plot(df_pd_noexo_set['time'], df_pd_noexo_set['solved'], 
           color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
           label='PD-NoExo' if set_num == 0 else '', marker='s', markersize=2)
    ax.plot(df_healthy_set['time'], df_healthy_set['solved'], 
           color=COLOR_HEALTHY, alpha=alpha_val, linewidth=2, 
           label='Healthy' if set_num == 0 else '', marker='^', markersize=2)

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Task Solved (1=Success, 0=Not Solved)', fontweight='bold')
ax.set_title('Task Success Timeline\nWhen Target Position is Reached', 
            fontweight='bold', pad=15)
ax.set_ylim([-0.1, 1.1])
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/11_task_success_timeline.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    if len(df_pd_exo_set) > 3:
        dt = df_pd_exo_set['time'].diff().mean()
        jerk_pd_exo = compute_jerk(df_pd_exo_set['jpos'].values, dt)
        jerk_pd_noexo = compute_jerk(df_pd_noexo_set['jpos'].values, dt)
        jerk_healthy = compute_jerk(df_healthy_set['jpos'].values, dt)
        
        alpha_val = 0.7 if EPI_NUM == 1 else 0.4
        
        ax.plot(df_pd_exo_set['time'], np.abs(jerk_pd_exo), 
               color=COLOR_PD_EXO, alpha=alpha_val, linewidth=1.5, 
               label='PD+Exo' if set_num == 0 else '')
        ax.plot(df_pd_noexo_set['time'], np.abs(jerk_pd_noexo), 
               color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=1.5, 
               label='PD-NoExo' if set_num == 0 else '')
        ax.plot(df_healthy_set['time'], np.abs(jerk_healthy), 
               color=COLOR_HEALTHY, alpha=alpha_val, linewidth=1.5, 
               label='Healthy' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Jerk Magnitude (rad/s³)', fontweight='bold')
ax.set_title('Movement Jerk Analysis\nLower Jerk = Smoother Motion', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/12_jerk_analysis_smoothness.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    
    if len(df_pd_exo_set) > 0:
        power_pd_exo = compute_power(df_pd_exo_set['exo_torque'].values, 
                                     df_pd_exo_set['jvel'].values)
        
        alpha_val = 0.7 if EPI_NUM == 1 else 0.5
        
        ax.plot(df_pd_exo_set['time'], np.abs(power_pd_exo), 
               color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
               label=f'Set {set_num+1}' if EPI_NUM > 1 else 'Exo Power')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Instantaneous Power (W)', fontweight='bold')
ax.set_title('Exoskeleton Power Consumption Over Time\n|Torque × Angular Velocity|', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/13_power_consumption.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

window_size = 20  # Rolling window for variance calculation

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    if len(df_pd_exo_set) > window_size:
        var_pd_exo = df_pd_exo_set['jvel'].rolling(window=window_size).var()
        var_pd_noexo = df_pd_noexo_set['jvel'].rolling(window=window_size).var()
        var_healthy = df_healthy_set['jvel'].rolling(window=window_size).var()
        
        alpha_val = 0.7 if EPI_NUM == 1 else 0.4
        
        ax.plot(df_pd_exo_set['time'], var_pd_exo, 
               color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
               label='PD+Exo' if set_num == 0 else '')
        ax.plot(df_pd_noexo_set['time'], var_pd_noexo, 
               color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
               label='PD-NoExo' if set_num == 0 else '')
        ax.plot(df_healthy_set['time'], var_healthy, 
               color=COLOR_HEALTHY, alpha=alpha_val, linewidth=2, 
               label='Healthy' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Velocity Variance (rad²/s²)', fontweight='bold')
ax.set_title(f'Movement Stability Analysis (Rolling Window = {window_size} steps)\nLower Variance = More Stable', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/14_velocity_variance_stability.png', dpi=300, bbox_inches='tight')
plt.close()


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

normalized_data = []
for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    if len(df_pd_exo_set) > 0:
        error_pd_exo_val = df_pd_exo_set['pose_err'].iloc[-1]
        error_pd_exo = np.mean(np.abs(error_pd_exo_val)) if hasattr(error_pd_exo_val, '__len__') else np.abs(error_pd_exo_val)
    else:
        error_pd_exo = 0
        
    if len(df_pd_noexo_set) > 0:
        error_pd_noexo_val = df_pd_noexo_set['pose_err'].iloc[-1]
        error_pd_noexo = np.mean(np.abs(error_pd_noexo_val)) if hasattr(error_pd_noexo_val, '__len__') else np.abs(error_pd_noexo_val)
    else:
        error_pd_noexo = 0
        
    if len(df_healthy_set) > 0:
        error_healthy_val = df_healthy_set['pose_err'].iloc[-1]
        error_healthy = np.mean(np.abs(error_healthy_val)) if hasattr(error_healthy_val, '__len__') else np.abs(error_healthy_val)
    else:
        error_healthy = 0
    
    if error_healthy > 1e-6:
        norm_pd_exo = (1 - (error_pd_exo / error_healthy)) * 100
        norm_pd_noexo = (1 - (error_pd_noexo / error_healthy)) * 100
    else:
        norm_pd_exo = 100
        norm_pd_noexo = 100
    
    normalized_data.append({
        'set': set_num,
        'PD+Exo': max(0, min(150, norm_pd_exo)),  
        'PD-NoExo': max(0, min(150, norm_pd_noexo)),
        'Healthy': 100
    })

df_norm = pd.DataFrame(normalized_data)

x_pos = np.arange(EPI_NUM)
width = 0.25

ax1.bar(x_pos - width, df_norm['PD+Exo'].values, width, label='PD+Exo', 
       color=COLOR_PD_EXO, alpha=0.8)
ax1.bar(x_pos, df_norm['PD-NoExo'].values, width, label='PD-NoExo', 
       color=COLOR_PD_NOEXO, alpha=0.8)
ax1.bar(x_pos + width, df_norm['Healthy'].values, width, label='Healthy', 
       color=COLOR_HEALTHY, alpha=0.8)

ax1.set_xlabel('Evaluation Set', fontweight='bold')
ax1.set_ylabel('Normalized Performance (%)', fontweight='bold')
ax1.set_title('Normalized Performance Comparison\n(Healthy Baseline = 100%)', 
             fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f'Set {i+1}' for i in range(EPI_NUM)])
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
ax1.axhline(y=100, color='black', linestyle='--', linewidth=1, alpha=0.5)

restoration = df_norm['PD+Exo'].mean()
improvement = restoration - df_norm['PD-NoExo'].mean()

ax2.bar(['PD-NoExo', 'PD+Exo', 'Healthy'], 
       [df_norm['PD-NoExo'].mean(), df_norm['PD+Exo'].mean(), 100],
       color=[COLOR_PD_NOEXO, COLOR_PD_EXO, COLOR_HEALTHY], alpha=0.8)
ax2.set_ylabel('Average Performance (%)', fontweight='bold')
ax2.set_title(f'Movement Restoration Summary\nImprovement: +{improvement:.1f}%', 
             fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=100, color='black', linestyle='--', linewidth=1, alpha=0.5)

for i, v in enumerate([df_norm['PD-NoExo'].mean(), df_norm['PD+Exo'].mean(), 100]):
    ax2.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/15_normalized_performance.png', dpi=300, bbox_inches='tight')
plt.close()


fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Use first set for overlay
df_pd_exo_0 = df_pd_exo[df_pd_exo['set'] == 0]
df_pd_noexo_0 = df_pd_noexo[df_pd_noexo['set'] == 0]
df_healthy_0 = df_healthy[df_healthy['set'] == 0]

# Position
ax1.plot(df_pd_exo_0['time'], df_pd_exo_0['jpos'], 
        color=COLOR_PD_EXO, linewidth=2.5, label='PD+Exo')
ax1.plot(df_pd_noexo_0['time'], df_pd_noexo_0['jpos'], 
        color=COLOR_PD_NOEXO, linewidth=2.5, label='PD-NoExo')
ax1.plot(df_healthy_0['time'], df_healthy_0['jpos'], 
        color=COLOR_HEALTHY, linewidth=2.5, label='Healthy')
ax1.set_ylabel('Position (rad)', fontweight='bold')
ax1.set_title('Comprehensive Performance Overlay: Position, Torque, and Error', 
             fontweight='bold', pad=15)
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Exoskeleton Torque
ax2.plot(df_pd_exo_0['time'], df_pd_exo_0['exo_torque'], 
        color=COLOR_PD_EXO, linewidth=2.5, label='Exo Torque')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
ax2.set_ylabel('Exo Torque (Nm)', fontweight='bold')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

# Position Error
ax3.plot(df_pd_exo_0['time'], np.abs(df_pd_exo_0['pose_err']), 
        color=COLOR_PD_EXO, linewidth=2.5, label='PD+Exo')
ax3.plot(df_pd_noexo_0['time'], np.abs(df_pd_noexo_0['pose_err']), 
        color=COLOR_PD_NOEXO, linewidth=2.5, label='PD-NoExo')
ax3.plot(df_healthy_0['time'], np.abs(df_healthy_0['pose_err']), 
        color=COLOR_HEALTHY, linewidth=2.5, label='Healthy')
ax3.set_xlabel('Time (s)', fontweight='bold')
ax3.set_ylabel('Position Error (rad)', fontweight='bold')
ax3.legend(loc='best')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/16_comprehensive_overlay.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    alpha_val = 0.7 if EPI_NUM == 1 else 0.4
    
    # Calculate average muscle activation across all muscles
    avg_ma_pd_exo = [np.mean(ma) for ma in df_pd_exo_set['MA']]
    avg_ma_pd_noexo = [np.mean(ma) for ma in df_pd_noexo_set['MA']]
    avg_ma_healthy = [np.mean(ma) for ma in df_healthy_set['MA']]
    
    ax.plot(df_pd_exo_set['time'], avg_ma_pd_exo, 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
           label='PD+Exo' if set_num == 0 else '')
    ax.plot(df_pd_noexo_set['time'], avg_ma_pd_noexo, 
           color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
           label='PD-NoExo' if set_num == 0 else '')
    ax.plot(df_healthy_set['time'], avg_ma_healthy, 
           color=COLOR_HEALTHY, alpha=alpha_val, linewidth=2, 
           label='Healthy' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Average Muscle Activation', fontweight='bold')
ax.set_title('Average Muscle Activation Over Time\nExoskeleton Reduces Required Muscle Effort', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/17_muscle_activation_comparison.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    
    alpha_val = 0.7 if EPI_NUM == 1 else 0.4
    
    avg_mr_pd_exo = [np.mean(mr) for mr in df_pd_exo_set['MR']]
    avg_mr_pd_noexo = [np.mean(mr) for mr in df_pd_noexo_set['MR']]
    
    ax.plot(df_pd_exo_set['time'], avg_mr_pd_exo, 
           color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
           label='PD+Exo' if set_num == 0 else '')
    ax.plot(df_pd_noexo_set['time'], avg_mr_pd_noexo, 
           color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
           label='PD-NoExo' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Average Muscle Recovery', fontweight='bold')
ax.set_title('Muscle Recovery Capacity Over Time\nExoskeleton Enables Better Recovery', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/18_muscle_recovery_analysis.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    
    if len(df_pd_exo_set) > 0:
        energy_pd_exo = np.cumsum(np.abs(df_pd_exo_set['exo_torque'].values * 
                                         df_pd_exo_set['jvel'].values))
        
        avg_ma_pd_noexo = np.array([np.mean(ma) for ma in df_pd_noexo_set['MA']])
        energy_pd_noexo = np.cumsum(avg_ma_pd_noexo * np.abs(df_pd_noexo_set['jvel'].values))
        
        alpha_val = 0.7 if EPI_NUM == 1 else 0.4
        
        ax.plot(df_pd_exo_set['time'], energy_pd_exo, 
               color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
               label='PD+Exo' if set_num == 0 else '')
        ax.plot(df_pd_noexo_set['time'], energy_pd_noexo, 
               color=COLOR_PD_NOEXO, alpha=alpha_val, linewidth=2, 
               label='PD-NoExo (Muscle Effort)' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Cumulative Energy (J)', fontweight='bold')
ax.set_title('Energy Efficiency Comparison\nExoskeleton vs. Muscle-Only Movement', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/19_energy_efficiency.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    
    if len(df_pd_exo_set) > 0:
        ctrl_magnitude = [np.linalg.norm(ctrl) for ctrl in df_pd_exo_set['ctrl']]
        
        alpha_val = 0.7 if EPI_NUM == 1 else 0.5
        
        ax.plot(df_pd_exo_set['time'], ctrl_magnitude, 
               color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
               label=f'Set {set_num+1}' if EPI_NUM > 1 else 'Control Signal')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Control Signal Magnitude', fontweight='bold')
ax.set_title('PPO Controller Stability Over Time\nControl Signal Magnitude', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/20_control_stability.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    if len(df_pd_exo_set) > 0:
        alpha_val = 0.7 if EPI_NUM == 1 else 0.4
        
        avg_mlen_pd_exo = [np.mean(mlen) for mlen in df_pd_exo_set['mlen']]
        avg_mlen_healthy = [np.mean(mlen) for mlen in df_healthy_set['mlen']]
        
        ax.plot(df_pd_exo_set['time'], avg_mlen_pd_exo, 
               color=COLOR_PD_EXO, alpha=alpha_val, linewidth=2, 
               label='PD+Exo' if set_num == 0 else '')
        ax.plot(df_healthy_set['time'], avg_mlen_healthy, 
               color=COLOR_HEALTHY, alpha=alpha_val, linewidth=2, 
               label='Healthy' if set_num == 0 else '')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Average Actuator Length (m)', fontweight='bold')
ax.set_title('Muscle Actuator Length Comparison\nBiomechanical Movement Pattern', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/21_actuator_length_analysis.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

def safe_pose_error(x):
    err_val = x['pose_err'].iloc[-1]
    if hasattr(err_val, '__len__'):
        return np.mean(np.abs(err_val))
    else:
        return np.abs(err_val)

metrics_summary = {
    'PD+Exo': {
        'final_error': df_pd_exo.groupby('set').apply(safe_pose_error).mean(),
        'avg_reward': df_pd_exo.groupby('set')['cumulative_reward'].last().mean(),
        'success_rate': df_pd_exo.groupby('set')['solved'].last().mean() * 100,
        'avg_fatigue': df_pd_exo['avg_fatigue'].mean()
    },
    'PD-NoExo': {
        'final_error': df_pd_noexo.groupby('set').apply(safe_pose_error).mean(),
        'avg_reward': df_pd_noexo.groupby('set')['cumulative_reward'].last().mean(),
        'success_rate': df_pd_noexo.groupby('set')['solved'].last().mean() * 100,
        'avg_fatigue': df_pd_noexo['avg_fatigue'].mean()
    },
    'Healthy': {
        'final_error': df_healthy.groupby('set').apply(safe_pose_error).mean(),
        'avg_reward': df_healthy.groupby('set')['cumulative_reward'].last().mean(),
        'success_rate': df_healthy.groupby('set')['solved'].last().mean() * 100,
        'avg_fatigue': df_healthy['avg_fatigue'].mean()
    }
}

conditions = ['PD+Exo', 'PD-NoExo', 'Healthy']
colors = [COLOR_PD_EXO, COLOR_PD_NOEXO, COLOR_HEALTHY]

# Final Position Error
final_errors = [metrics_summary[c]['final_error'] for c in conditions]
ax1.bar(conditions, final_errors, color=colors, alpha=0.8)
ax1.set_ylabel('Position Error (rad)', fontweight='bold')
ax1.set_title('Average Final Position Error\n(Lower is Better)', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(final_errors):
    ax1.text(i, v, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

# Average Cumulative Reward
avg_rewards = [metrics_summary[c]['avg_reward'] for c in conditions]
ax2.bar(conditions, avg_rewards, color=colors, alpha=0.8)
ax2.set_ylabel('Cumulative Reward', fontweight='bold')
ax2.set_title('Average Cumulative Reward\n(Higher is Better)', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(avg_rewards):
    ax2.text(i, v, f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

# Task Success Rate
success_rates = [metrics_summary[c]['success_rate'] for c in conditions]
ax3.bar(conditions, success_rates, color=colors, alpha=0.8)
ax3.set_ylabel('Success Rate (%)', fontweight='bold')
ax3.set_title('Task Success Rate\n(Higher is Better)', fontweight='bold')
ax3.set_ylim([0, 105])
ax3.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(success_rates):
    ax3.text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')

# Average Muscle Fatigue
avg_fatigues = [metrics_summary[c]['avg_fatigue'] for c in conditions]
ax4.bar(conditions, avg_fatigues, color=colors, alpha=0.8)
ax4.set_ylabel('Average Fatigue Level', fontweight='bold')
ax4.set_title('Average Muscle Fatigue\n(Lower is Better for PD)', fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(avg_fatigues):
    ax4.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/22_performance_metrics_summary.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(10, 8))


for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    
    if len(df_pd_exo_set) > 10:
        alpha_val = 0.6 if EPI_NUM == 1 else 0.3
        ax.scatter(np.abs(df_pd_exo_set['exo_torque']), 
                  np.abs(df_pd_exo_set['pose_err']),
                  c=df_pd_exo_set['time'], cmap='viridis', 
                  alpha=alpha_val, s=20, 
                  label=f'Set {set_num+1}' if EPI_NUM > 1 else None)

ax.set_xlabel('|Exoskeleton Torque| (Nm)', fontweight='bold')
ax.set_ylabel('|Position Error| (rad)', fontweight='bold')
ax.set_title('Exoskeleton Torque vs Position Error\nShows Adaptive Control Response', 
            fontweight='bold', pad=15)
if EPI_NUM > 1:
    ax.legend(loc='best', framealpha=0.9)
cbar = plt.colorbar(ax.collections[0], ax=ax)
cbar.set_label('Time (s)', fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('plots/23_torque_error_correlation.png', dpi=300, bbox_inches='tight')
plt.close()


if EPI_NUM != 0 and EPI_NUM != 1:
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    set_numbers = list(range(EPI_NUM))
    
    final_errors_pd_exo = []
    final_errors_pd_noexo = []
    final_errors_healthy = []
    
    for i in set_numbers:
        if len(df_pd_exo[df_pd_exo['set']==i]) > 0:
            err_val = df_pd_exo[df_pd_exo['set']==i].iloc[-1]['pose_err']
            final_errors_pd_exo.append(np.mean(np.abs(err_val)) if hasattr(err_val, '__len__') else np.abs(err_val))
        else:
            final_errors_pd_exo.append(0)
            
        if len(df_pd_noexo[df_pd_noexo['set']==i]) > 0:
            err_val = df_pd_noexo[df_pd_noexo['set']==i].iloc[-1]['pose_err']
            final_errors_pd_noexo.append(np.mean(np.abs(err_val)) if hasattr(err_val, '__len__') else np.abs(err_val))
        else:
            final_errors_pd_noexo.append(0)
            
        if len(df_healthy[df_healthy['set']==i]) > 0:
            err_val = df_healthy[df_healthy['set']==i].iloc[-1]['pose_err']
            final_errors_healthy.append(np.mean(np.abs(err_val)) if hasattr(err_val, '__len__') else np.abs(err_val))
        else:
            final_errors_healthy.append(0)
    
    width = 0.25
    x_pos = np.arange(len(set_numbers))
    
    ax1.bar(x_pos - width, final_errors_pd_exo, width, 
           label='PD+Exo', color=COLOR_PD_EXO, alpha=0.8)
    ax1.bar(x_pos, final_errors_pd_noexo, width, 
           label='PD-NoExo', color=COLOR_PD_NOEXO, alpha=0.8)
    ax1.bar(x_pos + width, final_errors_healthy, width, 
           label='Healthy', color=COLOR_HEALTHY, alpha=0.8)
    ax1.set_xlabel('Evaluation Set', fontweight='bold')
    ax1.set_ylabel('Final Position Error (rad)', fontweight='bold')
    ax1.set_title('Final Position Error Across Sets', fontweight='bold')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels([f'Set {i+1}' for i in set_numbers])
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    
    rewards_pd_exo = [df_pd_exo[df_pd_exo['set']==i]['cumulative_reward'].iloc[-1] 
                     if len(df_pd_exo[df_pd_exo['set']==i]) > 0 else 0 
                     for i in set_numbers]
    rewards_pd_noexo = [df_pd_noexo[df_pd_noexo['set']==i]['cumulative_reward'].iloc[-1] 
                       if len(df_pd_noexo[df_pd_noexo['set']==i]) > 0 else 0 
                       for i in set_numbers]
    rewards_healthy = [df_healthy[df_healthy['set']==i]['cumulative_reward'].iloc[-1] 
                      if len(df_healthy[df_healthy['set']==i]) > 0 else 0 
                      for i in set_numbers]
    
    ax2.bar(x_pos - width, rewards_pd_exo, width, 
           label='PD+Exo', color=COLOR_PD_EXO, alpha=0.8)
    ax2.bar(x_pos, rewards_pd_noexo, width, 
           label='PD-NoExo', color=COLOR_PD_NOEXO, alpha=0.8)
    ax2.bar(x_pos + width, rewards_healthy, width, 
           label='Healthy', color=COLOR_HEALTHY, alpha=0.8)
    ax2.set_xlabel('Evaluation Set', fontweight='bold')
    ax2.set_ylabel('Cumulative Reward', fontweight='bold')
    ax2.set_title('Cumulative Reward Across Sets', fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels([f'Set {i+1}' for i in set_numbers])
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    success_pd_exo = [df_pd_exo[df_pd_exo['set']==i]['solved'].iloc[-1] 
                     if len(df_pd_exo[df_pd_exo['set']==i]) > 0 else 0 
                     for i in set_numbers]
    success_pd_noexo = [df_pd_noexo[df_pd_noexo['set']==i]['solved'].iloc[-1] 
                       if len(df_pd_noexo[df_pd_noexo['set']==i]) > 0 else 0 
                       for i in set_numbers]
    success_healthy = [df_healthy[df_healthy['set']==i]['solved'].iloc[-1] 
                      if len(df_healthy[df_healthy['set']==i]) > 0 else 0 
                      for i in set_numbers]
    
    ax3.bar(x_pos - width, success_pd_exo, width, 
           label='PD+Exo', color=COLOR_PD_EXO, alpha=0.8)
    ax3.bar(x_pos, success_pd_noexo, width, 
           label='PD-NoExo', color=COLOR_PD_NOEXO, alpha=0.8)
    ax3.bar(x_pos + width, success_healthy, width, 
           label='Healthy', color=COLOR_HEALTHY, alpha=0.8)
    ax3.set_xlabel('Evaluation Set', fontweight='bold')
    ax3.set_ylabel('Task Solved (1=Yes, 0=No)', fontweight='bold')
    ax3.set_title('Task Success Across Sets', fontweight='bold')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels([f'Set {i+1}' for i in set_numbers])
    ax3.set_ylim([0, 1.1])
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')
    
    avg_torque_per_set = [df_pd_exo[df_pd_exo['set']==i]['exo_torque'].abs().mean() 
                          if len(df_pd_exo[df_pd_exo['set']==i]) > 0 else 0 
                          for i in set_numbers]
    
    ax4.bar(set_numbers, avg_torque_per_set, color=COLOR_PD_EXO, alpha=0.8)
    ax4.set_xlabel('Evaluation Set', fontweight='bold')
    ax4.set_ylabel('Average |Exo Torque| (Nm)', fontweight='bold')
    ax4.set_title('Average Exoskeleton Torque Across Sets', fontweight='bold')
    ax4.set_xticks(set_numbers)
    ax4.set_xticklabels([f'Set {i+1}' for i in set_numbers])
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('plots/24_multi_set_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()


print(f"  Average Final Error: {metrics_summary['PD+Exo']['final_error']:.6f} rad")
print(f"  Average  Reward: {metrics_summary['PD+Exo']['avg_reward']:.3f}")
print(f"  Task Success Rate: {metrics_summary['PD+Exo']['success_rate']:.1f}%")
print(f"  Average Muscle Fatigue: {metrics_summary['PD+Exo']['avg_fatigue']:.3f}")

print(f"  Average Final Error: {metrics_summary['PD-NoExo']['final_error']:.6f} rad")
print(f"  Average  Reward: {metrics_summary['PD-NoExo']['avg_reward']:.3f}")
print(f"  Task Success Rate: {metrics_summary['PD-NoExo']['success_rate']:.1f}%")
print(f"  Average Muscle Fatigue: {metrics_summary['PD-NoExo']['avg_fatigue']:.3f}")

print(f"  Average Final Error: {metrics_summary['Healthy']['final_error']:.6f} rad")
print(f"  Average  Reward: {metrics_summary['Healthy']['avg_reward']:.3f}")
print(f"  Task Success Rate: {metrics_summary['Healthy']['success_rate']:.1f}%")
print(f"  Average Muscle Fatigue: {metrics_summary['Healthy']['avg_fatigue']:.3f}")

print(f"  Average Latency: {avg_latency:.4f} ms")
print(f"  Min Latency: {df_latency['latency_ms'].min():.4f} ms")
print(f"  Max Latency: {df_latency['latency_ms'].max():.4f} ms")

error_improvement = ((metrics_summary['PD-NoExo']['final_error'] - 
                     metrics_summary['PD+Exo']['final_error']) / 
                     metrics_summary['PD-NoExo']['final_error']) * 100
reward_improvement = ((metrics_summary['PD+Exo']['avg_reward'] - 
                      metrics_summary['PD-NoExo']['avg_reward']) / 
                      abs(metrics_summary['PD-NoExo']['avg_reward'])) * 100

print(f"  Position Error Reduction: {error_improvement:.1f}%")
print(f"  Reward Improvement: {reward_improvement:.1f}%")
print(f"  Success Rate Improvement: {metrics_summary['PD+Exo']['success_rate'] - metrics_summary['PD-NoExo']['success_rate']:.1f}%")

restoration_percentage = (1 - (metrics_summary['PD+Exo']['final_error'] / 
                              metrics_summary['Healthy']['final_error'])) * 100
print(f"  Movement Restoration: {restoration_percentage:.1f}% of healthy performance")


def safe_pose_error(x):
    err_val = x['pose_err'].iloc[-1]
    if hasattr(err_val, '__len__'):
        return np.mean(np.abs(err_val))
    else:
        return np.abs(err_val)

summary_metrics = {
    'Condition': ['PD+Exo', 'PD-NoExo', 'Healthy'],
    'Total_Episodes': [len(df_pd_exo), len(df_pd_noexo), len(df_healthy)],
    'Avg_Cumulative_Reward': [
        df_pd_exo.groupby('set')['cumulative_reward'].last().mean(),
        df_pd_noexo.groupby('set')['cumulative_reward'].last().mean(),
        df_healthy.groupby('set')['cumulative_reward'].last().mean()
    ],
    'Avg_Final_Pose_Error': [
        df_pd_exo.groupby('set').apply(safe_pose_error).mean(),
        df_pd_noexo.groupby('set').apply(safe_pose_error).mean(),
        df_healthy.groupby('set').apply(safe_pose_error).mean()
    ],
    'Task_Success_Rate': [
        df_pd_exo.groupby('set')['solved'].last().mean() * 100,
        df_pd_noexo.groupby('set')['solved'].last().mean() * 100,
        df_healthy.groupby('set')['solved'].last().mean() * 100
    ]
}
df_summary = pd.DataFrame(summary_metrics)
df_summary.to_csv('data/summary_metrics.csv', index=False)

print(f"{'='*80}")

os.makedirs('plots', exist_ok=True)

plt.style.use('default')
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 9


fig, axes = plt.subplots(1, EPI_NUM, figsize=(6*EPI_NUM, 6))
if EPI_NUM == 1:
    axes = [axes]

recovery_summary = []

for set_num in range(EPI_NUM):
    ax = axes[set_num]
    
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    
    healthy_pos = df_healthy_set['jpos'].values
    pd_exo_pos = df_pd_exo_set['jpos'].values
    pd_noexo_pos = df_pd_noexo_set['jpos'].values
    
    min_len = min(len(healthy_pos), len(pd_exo_pos), len(pd_noexo_pos))
    healthy_pos = healthy_pos[:min_len]
    pd_exo_pos = pd_exo_pos[:min_len]
    pd_noexo_pos = pd_noexo_pos[:min_len]
    time_vals = df_healthy_set['time'].values[:min_len]
    
    healthy_accuracy = np.ones(min_len) * 100
    
    pd_exo_deviation = np.abs(pd_exo_pos - healthy_pos)
    pd_noexo_deviation = np.abs(pd_noexo_pos - healthy_pos)
    
    max_deviation = max(np.max(pd_exo_deviation), np.max(pd_noexo_deviation), 0.01)
    
    pd_exo_accuracy = 100 * (1 - pd_exo_deviation / max_deviation)
    pd_noexo_accuracy = 100 * (1 - pd_noexo_deviation / max_deviation)
    
    # Plot
    ax.plot(time_vals, healthy_accuracy, color=COLOR_HEALTHY, linewidth=3, 
            label='Healthy (100% Baseline)', linestyle='--', alpha=0.8)
    ax.plot(time_vals, pd_exo_accuracy, color=COLOR_PD_EXO, linewidth=2.5, 
            label='PD+Exo', alpha=0.9)
    ax.plot(time_vals, pd_noexo_accuracy, color=COLOR_PD_NOEXO, linewidth=2.5, 
            label='PD-NoExo', alpha=0.9)
    
    ax.set_xlabel('Time (s)', fontweight='bold')
    ax.set_ylabel('Movement Accuracy (%)', fontweight='bold')
    ax.set_title(f'Set {set_num+1}: Movement Recovery\nRelative to Healthy Baseline', 
                fontweight='bold')
    ax.legend(loc='best', framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 105])
    ax.axhline(y=100, color='black', linestyle=':', linewidth=1, alpha=0.5)
    
    avg_pd_exo = np.mean(pd_exo_accuracy)
    avg_pd_noexo = np.mean(pd_noexo_accuracy)
    
    textstr = f'Avg PD+Exo: {avg_pd_exo:.1f}%\nAvg PD-NoExo: {avg_pd_noexo:.1f}%\nImprovement: {avg_pd_exo - avg_pd_noexo:.1f}%'
    ax.text(0.02, 0.02, textstr, transform=ax.transAxes, fontsize=10,
            verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    recovery_summary.append({
        'set': set_num,
        'avg_pd_exo_accuracy': avg_pd_exo,
        'avg_pd_noexo_accuracy': avg_pd_noexo,
        'improvement': avg_pd_exo - avg_pd_noexo
    })

plt.tight_layout()
plt.savefig('plots/NEW_01_movement_recovery_per_episode.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

restoration_percentages = []

for set_num in range(EPI_NUM):
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    
    # Get position trajectories
    healthy_pos = df_healthy_set['jpos'].values
    pd_exo_pos = df_pd_exo_set['jpos'].values
    pd_noexo_pos = df_pd_noexo_set['jpos'].values
    
    min_len = min(len(healthy_pos), len(pd_exo_pos), len(pd_noexo_pos))
    healthy_pos = healthy_pos[:min_len]
    pd_exo_pos = pd_exo_pos[:min_len]
    pd_noexo_pos = pd_noexo_pos[:min_len]
    time_vals = df_healthy_set['time'].values[:min_len]
    
    healthy_movement = np.cumsum(np.abs(np.diff(healthy_pos, prepend=healthy_pos[0])))
    pd_exo_movement = np.cumsum(np.abs(np.diff(pd_exo_pos, prepend=pd_exo_pos[0])))
    pd_noexo_movement = np.cumsum(np.abs(np.diff(pd_noexo_pos, prepend=pd_noexo_pos[0])))
    
    restoration_pct = np.zeros(min_len)
    for i in range(min_len):
        if pd_noexo_movement[i] > 0.001:  # Safe threshold to avoid division by zero
            restoration_pct[i] = ((pd_exo_movement[i] - pd_noexo_movement[i]) / pd_noexo_movement[i]) * 100
        else:
            restoration_pct[i] = 0
    
    alpha_val = 0.8 if EPI_NUM == 1 else 0.6
    ax.plot(time_vals, restoration_pct, linewidth=2.5, alpha=alpha_val,
            label=f'Set {set_num+1}', color=plt.cm.RdYlGn(0.3 + 0.4*(set_num/max(EPI_NUM-1, 1))))
    
    avg_restoration = np.mean(restoration_pct[restoration_pct > -200])  # Filter outliers
    restoration_percentages.append(avg_restoration)

ax.axhline(y=0, color='black', linestyle='--', linewidth=1.5, alpha=0.7, 
          label='No Improvement (0%)')
ax.axhline(y=100, color='green', linestyle=':', linewidth=1.5, alpha=0.5, 
          label='100% More Movement')

ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Movement Restoration (%)', fontweight='bold')
ax.set_title('Movement Restoration: PD+Exo vs PD-NoExo\n(% Increase in Movement Achieved with Exoskeleton)', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3)

avg_restoration_all = np.mean(restoration_percentages)
ax.text(0.5, 0.95, f'Average Restoration: {avg_restoration_all:.1f}%', 
        transform=ax.transAxes, fontsize=12, fontweight='bold',
        horizontalalignment='center', verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

plt.tight_layout()
plt.savefig('plots/NEW_02_movement_restoration_improvement.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(12, 7))

speed_restoration_data = []

for set_num in range(EPI_NUM):
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    
    # Get velocity data
    healthy_vel = np.abs(df_healthy_set['jvel'].values)
    pd_exo_vel = np.abs(df_pd_exo_set['jvel'].values)
    pd_noexo_vel = np.abs(df_pd_noexo_set['jvel'].values)
    
    min_len = min(len(healthy_vel), len(pd_exo_vel), len(pd_noexo_vel))
    healthy_vel = healthy_vel[:min_len]
    pd_exo_vel = pd_exo_vel[:min_len]
    pd_noexo_vel = pd_noexo_vel[:min_len]
    time_vals = df_healthy_set['time'].values[:min_len]
    
    speed_restoration = np.zeros(min_len)
    for i in range(min_len):
        if pd_noexo_vel[i] > 0.001:  # Safe threshold
            speed_restoration[i] = ((pd_exo_vel[i] - pd_noexo_vel[i]) / pd_noexo_vel[i]) * 100
        else:
            speed_restoration[i] = 0
    
    alpha_val = 0.8 if EPI_NUM == 1 else 0.6
    ax.plot(time_vals, speed_restoration, linewidth=2.5, alpha=alpha_val,
            label=f'Set {set_num+1}', color=plt.cm.Reds(0.4 + 0.4*(set_num/max(EPI_NUM-1, 1))))
    
    avg_speed_rest = np.mean(speed_restoration[np.abs(speed_restoration) < 500])  # Filter outliers
    speed_restoration_data.append({
        'set': set_num,
        'avg_speed_restoration': avg_speed_rest,
        'max_speed_pd_exo': np.max(pd_exo_vel),
        'max_speed_pd_noexo': np.max(pd_noexo_vel),
        'avg_speed_pd_exo': np.mean(pd_exo_vel),
        'avg_speed_pd_noexo': np.mean(pd_noexo_vel)
    })

ax.axhline(y=0, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Speed Restoration (%)', fontweight='bold')
ax.set_title('Speed Restoration: How Much Faster PD+Exo Moves\n(% Increase in Velocity vs PD-NoExo)', 
            fontweight='bold', pad=15)
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3)

avg_speed_restoration = np.mean([d['avg_speed_restoration'] for d in speed_restoration_data])
ax.text(0.5, 0.95, f'Average Speed Restoration: {avg_speed_restoration:.1f}%', 
        transform=ax.transAxes, fontsize=12, fontweight='bold',
        horizontalalignment='center', verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='#FFB6C1', alpha=0.8))

plt.tight_layout()
plt.savefig('plots/NEW_03A_speed_restoration.png', dpi=300, bbox_inches='tight')
plt.close()


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

initiation_data = []
VELOCITY_THRESHOLD = 0.15  # rad/s threshold for movement initiation

for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    pd_exo_vel = np.abs(df_pd_exo_set['jvel'].values)
    pd_noexo_vel = np.abs(df_pd_noexo_set['jvel'].values)
    healthy_vel = np.abs(df_healthy_set['jvel'].values)
    
    initiation_pd_exo = detect_movement_initiation(pd_exo_vel, VELOCITY_THRESHOLD)
    initiation_pd_noexo = detect_movement_initiation(pd_noexo_vel, VELOCITY_THRESHOLD)
    initiation_healthy = detect_movement_initiation(healthy_vel, VELOCITY_THRESHOLD)
    
    dt = df_pd_exo_set['time'].iloc[1] - df_pd_exo_set['time'].iloc[0] if len(df_pd_exo_set) > 1 else 0.01
    time_pd_exo = initiation_pd_exo * dt
    time_pd_noexo = initiation_pd_noexo * dt
    time_healthy = initiation_healthy * dt
    
    initiation_data.append({
        'set': set_num + 1,
        'PD+Exo': time_pd_exo,
        'PD-NoExo': time_pd_noexo,
        'Healthy': time_healthy
    })

df_initiation = pd.DataFrame(initiation_data)

x_pos = np.arange(len(df_initiation))
width = 0.25

bars1 = ax1.bar(x_pos - width, df_initiation['PD+Exo'], width, 
                label='PD+Exo', color=COLOR_PD_EXO, alpha=0.8)
bars2 = ax1.bar(x_pos, df_initiation['PD-NoExo'], width, 
                label='PD-NoExo', color=COLOR_PD_NOEXO, alpha=0.8)
bars3 = ax1.bar(x_pos + width, df_initiation['Healthy'], width, 
                label='Healthy', color=COLOR_HEALTHY, alpha=0.8)

ax1.set_xlabel('Evaluation Set', fontweight='bold')
ax1.set_ylabel('Movement Initiation Time (s)', fontweight='bold')
ax1.set_title('Movement Initiation Speed\n(Lower = Faster Reaction)', fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f'Set {i+1}' for i in range(len(df_initiation))])
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}s', ha='center', va='bottom', fontsize=8)

avg_time_pd_exo = df_initiation['PD+Exo'].mean()
avg_time_pd_noexo = df_initiation['PD-NoExo'].mean()
avg_time_healthy = df_initiation['Healthy'].mean()

if avg_time_pd_noexo > 0.001:
    speedup_vs_noexo = ((avg_time_pd_noexo - avg_time_pd_exo) / avg_time_pd_noexo) * 100
else:
    speedup_vs_noexo = 0.0

if avg_time_healthy > 0.001:
    speedup_vs_healthy = ((avg_time_healthy - avg_time_pd_exo) / avg_time_healthy) * 100
else:
    speedup_vs_healthy = 0.0

conditions = ['vs PD-NoExo\n(Bradykinesia\nCompensation)', 'vs Healthy\n(Performance\nComparison)']
improvements = [speedup_vs_noexo, speedup_vs_healthy]
colors_imp = [COLOR_PD_EXO, COLOR_HEALTHY]

bars = ax2.bar(conditions, improvements, color=colors_imp, alpha=0.8, edgecolor='black', linewidth=2)
ax2.set_ylabel('Reaction Speed Improvement (%)', fontweight='bold')
ax2.set_title('PD+Exo Reaction Time Performance\n(Positive = Faster Response)', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=0, color='black', linestyle='--', linewidth=1)

for bar, val in zip(bars, improvements):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}%', ha='center', va='bottom' if val > 0 else 'top', 
            fontsize=14, fontweight='bold')

summary_text = f'Average Initiation Times:\n'
summary_text += f'PD+Exo: {avg_time_pd_exo:.3f}s\n'
summary_text += f'PD-NoExo: {avg_time_pd_noexo:.3f}s\n'
summary_text += f'Healthy: {avg_time_healthy:.3f}s\n\n'
summary_text += f'PD+Exo is {speedup_vs_noexo:.1f}% faster\nthan PD-NoExo'

ax2.text(0.5, 0.02, summary_text, transform=ax2.transAxes, fontsize=10,
        verticalalignment='bottom', horizontalalignment='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.tight_layout()
plt.savefig('plots/NEW_03B_reaction_activation_analysis.png', dpi=300, bbox_inches='tight')
plt.close()


fig, ax = plt.subplots(figsize=(10, 7))

avg_speeds = []
for set_num in range(EPI_NUM):
    df_pd_exo_set = df_pd_exo[df_pd_exo['set'] == set_num]
    df_pd_noexo_set = df_pd_noexo[df_pd_noexo['set'] == set_num]
    df_healthy_set = df_healthy[df_healthy['set'] == set_num]
    
    avg_speeds.append({
        'set': set_num,
        'PD+Exo': np.mean(np.abs(df_pd_exo_set['jvel'].values)),
        'PD-NoExo': np.mean(np.abs(df_pd_noexo_set['jvel'].values)),
        'Healthy': np.mean(np.abs(df_healthy_set['jvel'].values))
    })

df_speeds = pd.DataFrame(avg_speeds)

overall_avg = {
    'PD+Exo': df_speeds['PD+Exo'].mean(),
    'PD-NoExo': df_speeds['PD-NoExo'].mean(),
    'Healthy': df_speeds['Healthy'].mean()
}

conditions = ['PD-NoExo', 'PD+Exo', 'Healthy']
speeds = [overall_avg['PD-NoExo'], overall_avg['PD+Exo'], overall_avg['Healthy']]
colors = [COLOR_PD_NOEXO, COLOR_PD_EXO, COLOR_HEALTHY]

bars = ax.bar(conditions, speeds, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
ax.set_ylabel('Average Angular Velocity (rad/s)', fontweight='bold')
ax.set_title('Average Movement Speed Comparison\nDemonstrates Bradykinesia Compensation', 
            fontweight='bold', pad=15)
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, speeds):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

if overall_avg['PD-NoExo'] > 0.001:
    speed_improvement = ((overall_avg['PD+Exo'] - overall_avg['PD-NoExo']) / overall_avg['PD-NoExo']) * 100
else:
    speed_improvement = 0.0

ax.text(0.5, 0.95, f'Speed Improvement: {speed_improvement:.1f}%\nPD+Exo vs PD-NoExo', 
        transform=ax.transAxes, fontsize=13, fontweight='bold',
        horizontalalignment='center', verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))

plt.tight_layout()
plt.savefig('plots/NEW_03C_average_speed_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
